#### Project 3: Autonomous Media Inclusivity Research Agent
##### Day 4: LangGraph Integration Tests — Agent + FastAPI + N8N

**Goal:** Confirm the full LangGraph pipeline works end-to-end, now that N8N was already smoke-tested in Day 2 with a mock endpoint. Day 4 replaces the mock with the real ReAct agent and runs full research reports for all 4 monitored outlets.

---

#### What Day 4 demonstrates


run_research(company)          ← Day 3: ReAct agent + 4 tools
        |
        v
generate_report(company, text) ← Anthropic claude-sonnet-4-6 formats 11-section Markdown
        |
        v
generate_summary(...)          ← 1-2 sentence Notion column summary
        |
        v
_parse_report_metrics(report)  ← regex extracts scores + harm flags
        |
        v
ResearchResponse (Pydantic)    ← structured JSON — exactly what N8N receives
        |
        v
save_report(company, report)   ← local Markdown backup in /reports/


**Monitored outlets:** Al Jazeera English · The Guardian · NPR · New York Times

> **Day 2 → Day 4 handoff:** N8N was imported and connected to a mock endpoint in Day 2.
> Day 4 confirms the same N8N workflow works with the live LangGraph agent — no N8N changes needed.

In [ ]:
# CELL 1: Setup

import os
import sys
import json
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

load_dotenv()

print(f"Working directory: {os.getcwd()}")
print()

# Confirm all modules are importable before running anything expensive
from src.agents.media_agent import run_research
from src.report_generator import generate_report, generate_summary, save_report
from app import _parse_report_metrics

print("All modules imported successfully:")
print("  src.agents.media_agent  → run_research")
print("  src.report_generator    → generate_report, generate_summary, save_report")
print("  app                     → _parse_report_metrics")

##### Step 1: Run the ReAct Agent

The Guardian uses the Guardian Open Platform API (full article body text + bylines),
giving it richer primary source data than the other 3 outlets which use RSS.
This makes it a good first full test — it generates the most differentiated scores.

In [ ]:
# CELL 2: Run agent research on The Guardian
# Expected: 8-12 tool calls, 30-90 seconds
# Guardian routes through the Open Platform API for full article text — richer data than RSS.

COMPANY = "The Guardian"

print(f"Starting full pipeline for: {COMPANY}")
print("Step 1/4: Running ReAct agent (30-90s)...")
print()

research_result = run_research(COMPANY)

print()
print(f"Agent complete.")
print(f"  Messages in history : {len(research_result['messages'])}")
print(f"  Error               : {research_result['error']}")
print(f"  Analysis length     : {len(research_result['final_analysis'])} chars")

# Count tool calls made
tool_calls = [
    tc['name']
    for msg in research_result['messages']
    for tc in getattr(msg, 'tool_calls', [])
]
print(f"  Tool calls made     : {len(tool_calls)}")
print(f"  Tools used          : {list(dict.fromkeys(tool_calls))}")

##### Step 2: Generate the Structured Report

The raw research summary is passed to Anthropic claude-sonnet-4-6,
which formats it into the 11-section Markdown report.
Claude is used here (not gpt-4o-mini) because report generation requires
faithful formatting of existing evidence — not open-ended reasoning.

In [ ]:
# CELL 3: Format research into Markdown report

print("Step 2/4: Formatting report with Anthropic claude-sonnet-4-6...")
print()

report_md = generate_report(COMPANY, research_result["final_analysis"])

print(f"Report generated: {len(report_md)} characters, {len(report_md.splitlines())} lines")
print()
print("=" * 60)
print("REPORT PREVIEW (first 1500 chars):")
print("=" * 60)
print(report_md[:1500])
print("...")

##### Step 3: Extract Structured Metrics

N8N cannot read a Markdown document — it needs individual fields.
The `_parse_report_metrics()` function extracts scores and harm flags
from the report using regex, so they can be mapped to Notion columns.

In [ ]:
# CELL 4: Parse scores and harm flags from the report

print("Step 3/4: Parsing structured metrics from report...")
print()

metrics = _parse_report_metrics(report_md)

print(f"Overall score    : {metrics['overall_score']}/10")
print(f"Community scores : {metrics['community_scores']}")
print(f"Harm flags found : {len(metrics['harm_flags'])}")
for flag in metrics['harm_flags']:
    print(f"  ⚠️  {flag}")

# Generate the Notion summary column
print()
summary = generate_summary(COMPANY, report_md, metrics['overall_score'])
print(f"Notion summary   : {summary}")
print(f"Summary length   : {len(summary)} chars (limit 200)")

In [ ]:
# CELL 5: Save report to /reports/

print("Step 4/4: Saving report to disk...")
print()

report_path = save_report(COMPANY, report_md)

print(f"Saved: {report_path}")

import os
reports = os.listdir('reports')
print(f"\nAll reports in /reports/ ({len(reports)} total):")
for r in sorted(reports):
    size = os.path.getsize(f'reports/{r}')
    print(f"  {r}  ({size:,} bytes)")

##### Step 4: Confirm N8N Payload Structure

The FastAPI `/research` endpoint returns a `ResearchResponse` object.
Below we construct the same payload and verify every field that
the N8N workflow expects is present and correctly typed.

In [ ]:
# CELL 6: Build and inspect the full API payload
# This is exactly what N8N receives when it calls POST /research

from datetime import date

payload = {
    "status": "success",
    "company": COMPANY,
    "report_text": report_md,
    "summary": summary,
    "report_path": str(report_path),
    "overall_score": metrics["overall_score"],
    "harm_flags": metrics["harm_flags"],
    "community_scores": metrics["community_scores"],
    "date": date.today().isoformat(),
}

# N8N field mapping check
N8N_EXPECTED_FIELDS = {
    "company":           (str,  "Notion: Company title"),
    "overall_score":     (float, "Notion: Score column"),
    "summary":           (str,  "Notion: Summary column"),
    "report_text":       (str,  "Notion: Page body"),
    "harm_flags":        (list, "Notion: Harm flags column"),
    "community_scores":  (dict, "Notion: Community scores column"),
    "date":              (str,  "Notion: Date column"),
    "status":            (str,  "N8N: route to success/error branch"),
}

print("N8N Payload Field Validation")
print("=" * 55)

all_pass = True
for field, (expected_type, destination) in N8N_EXPECTED_FIELDS.items():
    value = payload.get(field)
    actual_type = type(value)
    ok = isinstance(value, expected_type) and (value != "" and value is not None)
    status = "PASS" if ok else "FAIL"
    if not ok:
        all_pass = False
    preview = str(value)[:50].replace("\n", " ") if value else "EMPTY"
    print(f"{status}  {field:<20} → {destination}")
    print(f"       value: {preview}")

print()
print("=" * 55)
print("ALL FIELDS VALID — payload ready for N8N" if all_pass else "SOME FIELDS INVALID — check above")

In [ ]:
# CELL 7: Print the full payload as JSON
# This is a redacted view (report_text truncated) matching what N8N logs show

display_payload = payload.copy()
display_payload["report_text"] = report_md[:300] + f"... [{len(report_md)} chars total]"

print("Full API response (report_text truncated for readability):")
print()
print(json.dumps(display_payload, indent=2))

##### Step 5: Test the FastAPI Endpoint Directly

Confirms the HTTP server returns the same payload when called via POST,
exactly as N8N will call it. Run `python run.py` in a terminal first.

In [ ]:
# CELL 8: Health check — confirm FastAPI is running
# Run `python run.py` in a terminal before executing this cell

import requests

try:
    resp = requests.get("http://localhost:8000/", timeout=3)
    print(f"FastAPI status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))
except requests.exceptions.ConnectionError:
    print("FastAPI not running.")
    print("Start the server: python run.py")
    print("Then re-run this cell.")

In [ ]:
# CELL 9: Call POST /research — exactly as N8N does
# Only run this if Cell 8 showed FastAPI is running
# This triggers a full research run (30-90s) via the HTTP server —
# confirming the same path N8N takes when it calls the endpoint.

# NPR: RSS with content:encoded (~850 chars body text + named bylines)
# A good second outlet to test — different data source from Guardian API
TEST_COMPANY = "NPR"

print(f"Calling POST http://localhost:8000/research with company={TEST_COMPANY}")
print("This triggers the full pipeline via HTTP (as N8N would)...")
print()

try:
    resp = requests.post(
        "http://localhost:8000/research",
        json={"company": TEST_COMPANY, "focus": "all"},
        timeout=180,  # 3 min timeout for full research
    )
    data = resp.json()

    print(f"Status code     : {resp.status_code}")
    print(f"Company         : {data.get('company')}")
    print(f"Overall score   : {data.get('overall_score')}/10")
    print(f"Summary         : {data.get('summary')}")
    print(f"Harm flags      : {data.get('harm_flags')}")
    print(f"Community scores: {data.get('community_scores')}")
    print(f"Report path     : {data.get('report_path')}")
    print(f"Report length   : {len(data.get('report_text', ''))} chars")

except requests.exceptions.ConnectionError:
    print("FastAPI not running — start with: python run.py")
except requests.exceptions.Timeout:
    print("Request timed out — the research pipeline may still be running.")
    print("Check the server terminal for progress.")

#### Day 4 Complete — Summary

##### What was demonstrated:
- Full end-to-end LangGraph pipeline runs correctly, confirming the Day 2 mock can be replaced with the real agent
- All 8 N8N payload fields are present, correctly typed, and non-empty
- FastAPI endpoint returns identical payload when called via HTTP POST (same path N8N takes)
- Reports saved to `/reports/` — sample reports now available for all 4 monitored outlets

##### Reports generated across all 4 monitored outlets:

| Outlet | Data source | Report file |
|---|---|---|
| Al Jazeera English | RSS (no bylines, ~105 char teasers) | `reports/al_jazeera_english_YYYY-MM-DD.md` |
| The Guardian | Guardian Open Platform API (full body + bylines) | `reports/the_guardian_YYYY-MM-DD.md` |
| NPR | RSS content:encoded (~850 chars + named bylines) | `reports/npr_YYYY-MM-DD.md` |
| New York Times | RSS (~130 char teasers, named bylines) | `reports/new_york_times_YYYY-MM-DD.md` |

### N8N End-to-End Status:
- ✅ N8N workflow imported and smoke-tested with mock endpoint (Day 2)
- ✅ LangGraph agent confirmed working with real research data (Day 3)
- ✅ FastAPI + LangGraph + N8N confirmed working together (Day 4)
- ☐ Enable schedule trigger (Monday 6am) for automated weekly runs

---

#### Project 3 — Complete File-by-File Reference

##### Entry Points

**`run.py`** — Start the server here. `python run.py` starts uvicorn on port 8000.

**`app.py`** — FastAPI server. Three routes:
- `GET /` → health check
- `POST /research` → what N8N calls (returns full JSON: report, scores, harm flags)
- `POST /analyse` → alias, same behaviour

Also contains `_parse_report_metrics()` — extracts the score table and ⚠️ flags from Markdown so N8N gets structured data. Regex handles optional `**bold**` markers around scores.

---

##### The Brain

**`src/agents/media_agent.py`** — The LangGraph ReAct agent. `run_research("The Guardian")` is the function everything calls. Creates a `gpt-4o-mini` agent with 4 tools, runs up to 30 iterations, returns the final AI message as `final_analysis`.

**`src/report_generator.py`** — Takes `final_analysis` and makes a second LLM call to `claude-sonnet-4-6` to format it into the 11-section Markdown report. Also has `generate_summary()` (1-2 sentence Notion column) and `save_report()` (saves to `reports/`).

**`src/rag_pipeline.py`** — The Pinecone connection. Embeds queries with `text-embedding-ada-002`, queries `media-inclusivity-index`, returns top 3 chunks. Do not modify — everything imports from it.

---

##### The 4 Tools

**`src/tools/rss_tool.py`** — Primary source evidence. Fetches today's RSS feed (30 articles), byline analysis, community keyword scan, language flags. Special routing: Guardian → Guardian Open Platform API for full body text. NPR → `content:encoded` (~850 chars). Thin articles (<150 chars) → URL fetch fallback.

**`src/tools/rag_tool.py`** — Wraps `rag_pipeline.rag_query()` as a LangChain `@tool`. Agent calls this 4+ times with different angle filters to pull benchmarks from Pinecone.

**`src/tools/newsapi_tool.py`** — Searches recent news about the outlet's diversity record (last 30 days, top 5 articles). Uses `NEWSAPI_KEY` from `.env`.

**`src/tools/wikipedia_tool.py`** — Free Wikipedia REST API. Company overview, ownership, editorial history. No API key needed.

**`src/tools/__init__.py`** — Exports all 4 tools so `media_agent.py` can do `from src.tools import ...`.

---

##### Configuration & Data

**`data/project_config.json`** — Single source of truth: outlet names, 4 research angles, 4 communities, search queries, report section structure, Pinecone index name, model names.

**`media_inclusivity_KB.md`** — Knowledge base source: 19 chunks of inclusivity standards and benchmarks. Needed for re-ingestion if Pinecone index is wiped.

**`.env`** — API keys (not committed): `OPENAI_API_KEY`, `PINECONE_API_KEY`, `NEWSAPI_KEY`, `GUARDIAN_API_KEY`, `ANTHROPIC_API_KEY`.

---

##### N8N

**`n8n/media_agent_workflow_updated.json`** — Import directly into N8N. Nodes:
1. **4 triggers** — Schedule (Mon 6am), Form (on-demand), Webhook, Manual
2. **Prepare Input** — normalises company name from any trigger source
3. **HTTP Request** — `POST https://<ngrok-url>/research` (300s timeout)
4. **Check If Success** — routes to error path if `status != "success"`
5. **Code in JavaScript** — parses `overall_score` from report text as fallback
6. **Success Output** — assembles clean payload fields
7. **Notion — Save Report** — single node, dynamic `Status` field:
   `score ≥ 7 → Good Standing | score ≥ 4 → Needs Review | score < 4 → Flagged for Action`
   Full report saved as 9 text blocks (covers reports up to 17,100 chars)
8. **Slack DM** — always fires, sends company + score + status + summary
9. **Respond Success / Respond Error** — webhook response

> **ngrok required**: N8N is cloud-hosted (`dlbosma.app.n8n.cloud`) and cannot reach `localhost`. Run `ngrok http 8000` in a second terminal before triggering the workflow.

**`n8n/README.md`** — Import instructions and credential setup.

---

##### Notebooks

| Notebook | Purpose |
|---|---|
| `day1_planning.ipynb` | Project design, outlet selection rationale, data quality table |
| `day2_rag.ipynb` | KB ingestion into Pinecone — run once, don't re-run unless wiping index |
| `day3_agent.ipynb` | Agent development, tool testing, ReAct trace |
| `day4_integration.ipynb` | FastAPI + N8N integration tests (this notebook) |
| `report_prompts.ipynb` | The 4 report prompts — full text + design rationale |


---

#### Report Prompt Design Guide

> Full prompts live in **`notebooks/report_prompts.ipynb`**.
> This section is a reference summary — what each prompt does, why 4 steps, and what to pass in.

---

##### Why 4 prompts instead of 1?

Asking one LLM call to write the entire report in one shot produces generic filler and hallucination.
Breaking the task into 4 focused steps — each feeding into the next — produces dramatically better output:
each call is short, grounded, and has a single job.

```
Step 1: ANGLE_PROMPT       → writes the 4 evaluation angle sections (prose, evidence-cited)
Step 2: COMMUNITY_PROMPT   → synthesises across angles, one paragraph per community
Step 3: SYNTHESIS_PROMPT   → assigns 1–10 scores, identifies strengths, raises harm flags
Step 4: REPORT_PROMPT      → assembles the complete 11-section Markdown report
```

---

##### Step 1 — ANGLE_PROMPT

**Input:** `company`, `date`, `tool_results` (all 4 tool outputs concatenated)

**Output:** 4 prose sections (2–4 sentences each), every claim cited in `[square brackets]`

**Persona:** Nova_Tracks, lead researcher at Media Diversity Watch

**Key rules enforced in the prompt:**
- Full sentences only — no bullet points inside sections
- Every statistic must have a source citation: `"4 of 28 articles [RSS Feed: BBC, {date}]"`
- If no evidence was found for an angle, the exact phrase `No evidence found in this monitoring period.` is required
- LLM is explicitly forbidden from using training data — only what the tools returned

---

##### Step 2 — COMMUNITY_PROMPT

**Input:** `company`, `date`, `angle_analysis` (Step 1 output), `tool_results`

**Output:** 4 paragraphs (3–5 sentences each) — one per community

**Why this step exists:** The angle sections are organised by *what we measure* (bylines, sourcing, etc.).
This step reorganises the same evidence by *who we're looking at* (women, LGBTQ+, race, disability) —
a different lens that answers "what did this week mean specifically for this group?"

**Key rule:** This is a synthesis, not a copy — the prompt explicitly forbids repeating the angle text word for word.

---

##### Step 3 — SYNTHESIS_PROMPT

**Input:** `company`, `angle_analysis`, `community_summaries` (Step 2), `tool_results`

**Output:** Structured block with three parts:

```
SCORES:
Women and Gender Equality: 5/10 -- [one sentence reasoning with source]
LGBTQ+ Communities: 3/10 -- ...
Racial and Ethnic Minorities: 6/10 -- ...
People with Disabilities: 4/10 -- ...
Overall: 4.5/10

STRENGTHS:
[2–4 sentences of prose — must be specific, must cite evidence]

HARM FLAGS:
Flag: "[exact quote]" -- [why flagged]
```

**Scoring guide in the prompt:**
| Score | Meaning |
|---|---|
| 1–3 | Significant concerns or near absence |
| 4–6 | Mixed — some representation, notable gaps |
| 7–9 | Generally positive with minor issues |
| 10 | Exemplary (rare — requires strong evidence across all angles) |

**The Overall score** from this output is what `_parse_report_metrics()` in `app.py` extracts and sends to Notion and Slack.

---

##### Step 4 — REPORT_PROMPT

**Input:** `company`, `week_number`, `year`, `wikipedia_result`, `angle_analysis`, `community_summaries`, `synthesis`, `sources`

**Output:** Complete 11-section Markdown report, saved to `reports/`

**New sections written only here** (need the full picture to write):
- **Section 1: Company Overview** — from Wikipedia background (ownership, editorial stance, any stated diversity commitments)
- **Section 10: Recommendations** — 3–5 specific, evidence-grounded actions, each citing a finding from this report

**Key rule for recommendations:** Not generic ("improve diversity") but specific and sourced:
`"Increase female bylines in political coverage, where the current ratio is 14% against the 37% industry benchmark [Nieman Lab 2023]."`

---

##### Input/Output Chain at a Glance

```
tool_results (RSS + RAG + NewsAPI + Wikipedia)
    │
    ▼
ANGLE_PROMPT → angle_analysis
    │
    ▼
COMMUNITY_PROMPT(angle_analysis + tool_results) → community_summaries
    │
    ▼
SYNTHESIS_PROMPT(angle_analysis + community_summaries + tool_results) → synthesis (scores + flags)
    │
    ▼
REPORT_PROMPT(all of the above + wikipedia_result) → full 11-section Markdown
    │
    ▼
_parse_report_metrics() → overall_score, community_scores, harm_flags → N8N payload
```

---

##### Hallucination Prevention

Two mechanisms work together:

1. **Citation requirement** — every fact must be followed by `[Source: ...]`. If the LLM can't cite it, it shouldn't say it.
2. **Explicit training data ban** — every prompt includes: `"Do not use your training data. Only use the evidence provided below."`

This grounds the report in what the tools actually returned this week, not what the model knows in general about the outlet.
